In [1]:
#!/usr/bin/env python3
"""Merge Griffin result folders that share the same structure
(sample/sample.<correction>.coverage.tsv) into one folder, by appending
the rows (one row per TF site set) instead of overwriting files."""

import os
import sys
import pandas as pd

# ---- 1. Settings: edit these paths if needed ----
base = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling"
input_dirs = [
    os.path.join(base, "results"),
    os.path.join(base, "results_merged_44")
]
output_dir = os.path.join(base, "results_merged")   # no date in the name
corrections = ["GC_corrected", "uncorrected"]       # the two file types per sample

# ---- 2. Collect all sample names found in any input folder ----
samples = set()
for d in input_dirs:
    if not os.path.isdir(d):
        sys.exit(f"ERROR: folder not found: {d}")
    for name in os.listdir(d):
        if os.path.isdir(os.path.join(d, name)):
            samples.add(name)
samples = sorted(samples)
print(f"Found {len(samples)} samples across {len(input_dirs)} folders")

# ---- 3. Merge sample by sample, file type by file type ----
summary = []   # one line per merged file, for the final check
for sample in samples:
    for corr in corrections:
        frames = []
        for d in input_dirs:
            f = os.path.join(d, sample, f"{sample}.{corr}.coverage.tsv")
            if os.path.isfile(f):
                # dtype=str keeps the numbers exactly as written in the file
                df = pd.read_csv(f, sep="\t", dtype=str)
                df["_source"] = os.path.basename(d)   # temporary, to report duplicates
                frames.append(df)

        if not frames:
            print(f"WARNING: no {corr} file for {sample}")
            continue

        # Check that all files have the same columns (same bins and settings)
        ref_cols = [c for c in frames[0].columns if c != "_source"]
        for df in frames[1:]:
            cols = [c for c in df.columns if c != "_source"]
            if cols != ref_cols:
                sys.exit(f"ERROR: column mismatch for {sample} ({corr}), "
                         f"source {df['_source'].iloc[0]}")

        merged = pd.concat(frames, ignore_index=True)

        # Report and remove duplicated TF site sets (keep the first one)
        dup = merged[merged.duplicated("site_name", keep="first")]
        if len(dup) > 0:
            print(f"NOTE: {sample} ({corr}) duplicated site_name dropped: "
                  f"{sorted(dup['site_name'].unique())} "
                  f"(from {sorted(dup['_source'].unique())})")
            merged = merged.drop_duplicates("site_name", keep="first")

        merged = merged.drop(columns="_source")

        # Write the merged file in the same structure as the original
        out_sample_dir = os.path.join(output_dir, sample)
        os.makedirs(out_sample_dir, exist_ok=True)
        out_file = os.path.join(out_sample_dir, f"{sample}.{corr}.coverage.tsv")
        merged.to_csv(out_file, sep="\t", index=False)

        summary.append({"sample": sample, "correction": corr,
                        "n_tfs": len(merged),
                        "tfs": ",".join(sorted(merged["site_name"]))})

# ---- 4. Final check: do all samples have the same TFs? ----
summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(output_dir, "merge_summary.tsv"), sep="\t", index=False)

print("\nNumber of TF site sets per file (should be the same for all samples):")
print(summary_df.groupby("correction")["n_tfs"].value_counts())
print(f"\nDone. Merged files are in: {output_dir}")

/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Found 769 samples across 2 folders

Number of TF site sets per file (should be the same for all samples):
correction    n_tfs
GC_corrected  65       769
uncorrected   65       769
Name: count, dtype: int64

Done. Merged files are in: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling/results_merged


In [2]:
#!/usr/bin/env python3
"""Summarize Griffin GC-corrected coverage files for all TFs:
  - All_TFBS sheet: every TF once (master table)
  - Module_Map sheet: TF -> modules (TFs can be in several), role, gene-set source
  - one sheet per functional module (views; shared TFs are repeated)."""

import os
import sys
from collections import OrderedDict

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------
# 1. Settings (edit paths here)
# ---------------------------------------------------------------
BASE = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling"

# Folders with merged results. A folder that does not exist is skipped.
# If the same sample + TF appears in two folders, the first one is kept and
# the duplicate is counted in the QC sheet.
INPUT_DIRS = [
    os.path.join(BASE, "results_merged"),      # current merged results
    os.path.join(BASE, "results_merged_44"),   # earlier 44-TF results
]
OUT_DIR = BASE                                 # where the workbook is saved
CORRECTION = "GC_corrected"
METRICS = ["mean_coverage", "central_coverage", "amplitude"]

# ---------------------------------------------------------------
# 2. Functional modules. A TF may appear in several modules.
#    TFs in the data but in no module go to an "Unassigned" sheet.
# ---------------------------------------------------------------
MODULES = OrderedDict([
    ("ER_Pioneer_Luminal", ["ESR1", "FOXA1", "GATA3", "GRHL2", "TFAP2C", "TRPS1",
                            "PBX1", "SPDEF", "HOXB13", "ELF3"]),
    ("ER_Target_Response", ["PGR", "MYB", "XBP1", "EGR3", "FOS", "MYC", "RARA"]),
    ("ER_Negative_Basal", ["BCL11A", "TCF7L1", "HMGA1", "NFIL3", "TP63", "SOX9", "ZEB1"]),
    ("Hippo_EMT_Invasion", ["TEAD1", "TEAD4", "ETS1", "ZEB1", "SOX9"]),
    ("Nuclear_Receptor_Crosstalk", ["AR", "ESR2", "ESRRA", "NR3C1", "PGR", "RARA"]),
    ("Proliferation_CellCycle", ["E2F1", "E2F7", "FOXM1", "MYC", "TP53"]),
    ("AP1_Stress", ["JUN", "FOS", "FOSL1", "BACH1", "HIF1A", "CREB1"]),
    ("Growth_Survival_Signaling", ["FOXO1", "FOXO3", "NFKB1", "RELA", "STAT3"]),
    ("Interferon_HLA", ["IRF1", "STAT1", "STAT2", "RFX1", "RFX5", "NFYA", "NFYC"]),
    ("Bone_TGF_BMP", ["RUNX2", "SMAD1", "SMAD3", "SMAD4", "JUN"]),
    ("Blood_Immune_Control", ["SPI1", "RUNX1", "CEBPB", "IKZF1", "NFATC1"]),
    ("Unrelated_Control", ["HNF4A", "CDX2", "MYOD1"]),
    ("Technical_Control", ["CTCF"]),
])

# Role of each TF in the ER hypothesis (labels only, used later)
ROLE = {}
for t in ["ESR1", "FOXA1", "GATA3", "XBP1", "PGR", "MYB", "PBX1", "RARA"]:
    ROLE[t] = "ER_positive_primary"
for t in ["FOS", "EGR3", "MYC"]:
    ROLE[t] = "ER_positive_secondary"      # in er_upreg but multi-functional
for t in ["BCL11A", "TCF7L1", "HMGA1", "NFIL3"]:
    ROLE[t] = "ER_negative_primary"
for t in ["SPI1", "RUNX1", "IKZF1"]:
    ROLE[t] = "Control_blood"
for t in ["HNF4A", "CDX2", "MYOD1"]:
    ROLE[t] = "Control_unrelated"
ROLE["CTCF"] = "Control_technical"

# Source gene set (from the ER scoring workbook)
GENE_SET = {
    "FOS": "er_upreg", "MYB": "er_upreg;er_pos_corr", "PGR": "er_upreg",
    "EGR3": "er_upreg", "MYC": "er_upreg",
    "FOXA1": "er_pos_corr", "GATA3": "er_pos_corr", "XBP1": "er_pos_corr",
    "PBX1": "er_pos_corr", "RARA": "er_pos_corr",
    "BCL11A": "er_neg_corr", "HMGA1": "er_neg_corr",
    "NFIL3": "er_neg_corr", "TCF7L1": "er_neg_corr",
    "ESR1": "ESR1 itself",
}

# ---------------------------------------------------------------
# 3. Read every coverage file (only the columns we need)
# ---------------------------------------------------------------
needed = {"site_name", "correction", "sample"} | set(METRICS)
frames = []
sample_folders = set()
n_missing = n_empty = n_failed = n_read = n_invalid = 0
used_dirs = 0

for d in INPUT_DIRS:
    if not os.path.isdir(d):
        print(f"WARNING: folder not found, skipped: {d}")
        continue
    used_dirs += 1
    dir_files, dir_tfs = 0, set()
    for sample in sorted(os.listdir(d)):
        sdir = os.path.join(d, sample)
        if not os.path.isdir(sdir):
            continue
        sample_folders.add(sample)
        f = os.path.join(sdir, f"{sample}.{CORRECTION}.coverage.tsv")
        if not os.path.isfile(f):
            n_missing += 1
            continue
        if os.path.getsize(f) == 0:
            n_empty += 1
            continue
        try:
            df = pd.read_csv(f, sep="\t", usecols=lambda c: c in needed)
        except Exception as e:
            print(f"FAILED to read {f}: {e}")
            n_failed += 1
            continue
        if df.empty:
            n_empty += 1
            continue
        n_read += 1
        dir_files += 1
        df = df[df["correction"] == CORRECTION].copy()
        df["sample"] = sample                                   # folder name = sample ID
        df["TFBS"] = df["site_name"].str.replace("^site_", "", regex=True)
        for m in METRICS:
            df[m] = pd.to_numeric(df[m], errors="coerce")       # bad values -> NaN
        bad = df[METRICS].isna().any(axis=1)
        n_invalid += int(bad.sum())
        dir_tfs |= set(df.loc[~bad, "TFBS"])
        frames.append(df.loc[~bad, ["sample", "TFBS"] + METRICS])
    print(f"{os.path.basename(d)}: {dir_files} files read, {len(dir_tfs)} TFs")

if not frames:
    sys.exit("ERROR: no usable coverage files found")

data = pd.concat(frames, ignore_index=True)
n_dup = int(data.duplicated(["sample", "TFBS"]).sum())
data = data.drop_duplicates(["sample", "TFBS"], keep="first")

samples = sorted(data["sample"].unique())
tfs_in_data = set(data["TFBS"].unique())
n_tf = len(tfs_in_data)
wide = data.pivot(index="sample", columns="TFBS", values=METRICS).reindex(samples)
counts = data.groupby("TFBS")["sample"].nunique()

# Modules limited to TFs that exist in the data; leftovers -> Unassigned
modules = OrderedDict()
for name, tf_list in MODULES.items():
    present = [t for t in tf_list if t in tfs_in_data]
    if present:
        modules[name] = present
assigned = {t for tf_list in modules.values() for t in tf_list}
leftover = sorted(tfs_in_data - assigned)
if leftover:
    modules["Unassigned"] = leftover
    print("NOTE: TFs without a module (sheet 'Unassigned'):", leftover)

# Master TF order = order of first appearance across modules
all_tfs = []
for tf_list in modules.values():
    for t in tf_list:
        if t not in all_tfs:
            all_tfs.append(t)

# ---------------------------------------------------------------
# 4. Write the workbook
# ---------------------------------------------------------------
wb = Workbook()
blue = PatternFill("solid", fgColor="1F4E78")
orange = PatternFill("solid", fgColor="C55A11")
green = PatternFill("solid", fgColor="548235")
white_bold = Font(bold=True, color="FFFFFF")
center = Alignment(horizontal="center", vertical="center")


def style_header(cells, fill):
    for c in cells:
        c.font, c.fill, c.alignment = white_bold, fill, center


def write_wide(ws, tf_list, fill):
    """Two-row header (TF merged over its metrics), one row per sample."""
    ws.cell(1, 1, "sample")
    ws.merge_cells(start_row=1, start_column=1, end_row=2, end_column=1)
    col = 2
    for tf in tf_list:
        ws.cell(1, col, tf)
        ws.merge_cells(start_row=1, start_column=col, end_row=1, end_column=col + len(METRICS) - 1)
        for j, m in enumerate(METRICS):
            ws.cell(2, col + j, f"{tf}_{m}")
        col += len(METRICS)
    for r in (1, 2):
        style_header([ws.cell(r, c) for c in range(1, col)], fill)
    for i, s in enumerate(samples, start=3):
        ws.cell(i, 1, s)
        c = 2
        for tf in tf_list:
            for m in METRICS:
                v = wide.at[s, (m, tf)]
                if pd.notna(v):
                    ws.cell(i, c, float(v)).number_format = "0.00000"
                c += 1
    ws.freeze_panes = "B3"
    ws.column_dimensions["A"].width = 18
    for c in range(2, col):
        ws.column_dimensions[get_column_letter(c)].width = 20


# ---- QC_Summary ----
ws = wb.active
ws.title = "QC_Summary"
ws["A1"] = f"Griffin {n_tf}-TFBS Summary QC"
ws.merge_cells("A1:D1")
style_header([ws["A1"]], blue)
ws.append(["Metric", "Value"])
style_header(ws[2], blue)
for row in [
    ("Sample folders found", len(sample_folders)),
    ("Coverage files read", n_read),
    ("Samples represented", len(samples)),
    ("Configured TFBSs", n_tf),
    ("Missing coverage files", n_missing),
    ("Empty coverage files", n_empty),
    ("Failed files", n_failed),
    ("Invalid numeric rows removed", n_invalid),
    ("Duplicate sample-TFBS rows removed", n_dup),
]:
    ws.append(list(row))
ws.append([])
ws.append([])
hdr = ws.max_row + 1
ws.append(["Module", "TFBS", "Samples_with_result", "Missing_vs_completed_samples"])
style_header(ws[hdr], blue)
for mod, tf_list in modules.items():
    for tf in tf_list:
        n = int(counts.get(tf, 0))
        ws.append([mod, tf, n, len(samples) - n])
ws.freeze_panes = "A3"
ws.column_dimensions["A"].width = 34
for col in "BCD":
    ws.column_dimensions[col].width = 30

# ---- Module_Map: one row per TF (shared TFs list all their modules) ----
mm = wb.create_sheet("Module_Map")
mm.append(["TFBS", "Modules", "N_modules", "Role_in_ER_hypothesis", "Source_gene_set",
           "Samples_with_result"])
style_header(mm[1], blue)
for tf in all_tfs:
    mods = [m for m, lst in modules.items() if tf in lst]
    mm.append([tf, ";".join(mods), len(mods), ROLE.get(tf, "Other"),
               GENE_SET.get(tf, ""), int(counts.get(tf, 0))])
mm.freeze_panes = "A2"
for col, w in zip("ABCDEF", [14, 60, 12, 26, 24, 20]):
    mm.column_dimensions[col].width = w

# ---- All_TFBS: every TF once (master table) ----
write_wide(wb.create_sheet("All_TFBS"), all_tfs, green)

# ---- one sheet per module (views) ----
for mod, tf_list in modules.items():
    write_wide(wb.create_sheet(mod[:31]), tf_list, orange)

out = os.path.join(OUT_DIR, f"Griffin_{n_tf}TF_{CORRECTION}_summary_by_module.xlsx")   # no date in the name
wb.save(out)

shared = [t for t in all_tfs if sum(t in l for l in modules.values()) > 1]
print(f"\nSamples: {len(samples)} | TFs: {n_tf} | modules: {len(modules)} | sheets: {len(wb.sheetnames)}")
print(f"TFs in more than one module ({len(shared)}): {shared}")
print(f"Missing files: {n_missing} | empty: {n_empty} | failed: {n_failed} | invalid rows: {n_invalid} | duplicates: {n_dup}")
print(f"Saved: {out}")

results_merged: 769 files read, 65 TFs

Samples: 769 | TFs: 65 | modules: 13 | sheets: 16
TFs in more than one module (7): ['PGR', 'FOS', 'MYC', 'RARA', 'SOX9', 'ZEB1', 'JUN']
Missing files: 0 | empty: 0 | failed: 0 | invalid rows: 0 | duplicates: 0
Saved: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling/Griffin_65TF_GC_corrected_summary_by_module.xlsx
